In [1]:
# TABLE SEPARATE

import pymupdf

file = pymupdf.open(r'D:\LegalSaathi AI\data\pdfs\CrPC_2023.pdf')

table_file = pymupdf.open()

table_file.insert_pdf(file, from_page=174, to_page=203)

table_file.save('D:/LegalSaathi AI/src/notebooks/docs/bnss/table_bnss_174_204.pdf')
table_file.close()

# sections separate remvoe the tables

sections_file = pymupdf.open()

sections_file.insert_pdf(file,from_page= 17, to_page = 173)

sections_file.save('D:/LegalSaathi AI/src/notebooks/docs/bnss/sections_bnss_17_174.pdf')
sections_file.close()

In [23]:
# crop the file from left and right

sections_file = pymupdf.open(r'D:/LegalSaathi AI/src/notebooks/docs/bnss/sections_bnss_17_174.pdf')

for page in sections_file:
    rect = page.rect
    
    new_rect = pymupdf.Rect(
    rect.x0 + 110,  # Trim Left
    rect.y0 ,  # Trim Top
    rect.x1 - 110,  # Trim Right
    rect.y1    # Trim Bottom
    )
    
    page.set_cropbox(new_rect)


sections_file.save('D:/LegalSaathi AI/src/notebooks/docs/bnss/final_cropped_sections_bnss_17_174.pdf')


### Sections cleaning

In [ ]:
# CONVERT THE FILE INTO MD TEXT PARSER CODE RUNNED INTO THE KAGGGLE

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False                  # skip OCR = much faster
pipeline_options.do_table_structure = False        # detect tables
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=4,
    device=AcceleratorDevice.CUDA                 # use Kaggle GPU
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

result = converter.convert("/kaggle/input/datasets/rajgurubhosale/cropped-bns-sections/final_cropped_sections_bnss_17_174.pdf")

md = result.document.export_to_markdown()

with open("/kaggle/working/bnss_sections.md", "w") as f:
    f.write(md)

print("Done! output.md created")

### table cleaning

In [ ]:
# this code is runned into the kaggle

import pandas as pd
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True        # keep TRUE for tables!
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=4,
    device=AcceleratorDevice.CUDA
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

result = converter.convert("/kaggle/input/datasets/rajgurubhosale/bnss-tables/table_bnss_174_204.pdf")

# collect all tables into one dataframe
all_dfs = []
for table in result.document.tables:
    df = table.export_to_dataframe()
    all_dfs.append(df)

final_df = pd.concat(all_dfs, ignore_index=True)

# save as CSV (best for RAG pipeline)
final_df.to_csv("/kaggle/working/bnss_tables.csv", index=False)
                        
# save as markdown (for preview)
with open("/kaggle/working/bnss_tables.md", "w") as f:
    for table in result.document.tables:
        f.write(table.export_to_markdown())
        f.write("\n\n")

print(f"Done! {len(all_dfs)} tables found")
print(final_df.head())

In [1]:
# table cleaning

import pandas as pd

df = pd.read_csv(r'D:\LegalSaathi AI\src\notebooks\docs\bnss\bnss_tables.csv')

In [2]:
temp_df_1 = df.loc[df.iloc[:,:6].notna().all(axis=1)]
clean_df_1 = temp_df_1.dropna(axis=1,how='all')

In [3]:
temp_df_1 = df.loc[df.iloc[:,12:-2].notna().all(axis=1)]
clean_df_2 = temp_df_1.dropna(axis=1,how='all')

In [4]:
clean_df_2.columns = clean_df_1.columns

In [5]:
# final cleaned table
final_df  = pd.concat([clean_df_1, clean_df_2], axis=0, ignore_index=True)

In [6]:
final_df.head()

,Section,Offence,Punishment,Cognizable or non- cognizable,Bailable or Non- bailable,By what Court triable
0,1.0,2,3,4,5,6
1,49.0,Punishment of abetment if the act abetted is c...,Same as for offence abetted.,According as offence abetted is cognizable or ...,According as offence abetted is bailable or no...,Court by which offence abetted is triable.
2,50.0,Punishment of abetment if person abetted does ...,Ditto,Ditto,Ditto,Ditto.
3,51.0,Liability of abettor when one act abetted and ...,Same as for offence intended to be abetted.,Ditto,Ditto,Ditto.
4,52.0,Abettor when liable to cumulative punishment f...,Same as for offence committed.,Ditto,Ditto,Ditto


In [175]:
final_df.to_csv('D:/LegalSaathi AI/src/notebooks/docs/bnss/final_cleaned_df.csv',index=False)

# CHUNKING

## table chunking

In [36]:
import pandas as pd


df  = pd.read_csv(r'D:\LegalSaathi AI\data\output\bnss\final_cleaned_df.csv')
df.head()

,Section,Offence,Punishment,Cognizable or non- cognizable,Bailable or Non- bailable,By what Court triable
0,1.0,2,3,4,5,6
1,49.0,Punishment of abetment if the act abetted is c...,Same as for offence abetted.,According as offence abetted is cognizable or ...,According as offence abetted is bailable or no...,Court by which offence abetted is triable.
2,50.0,Punishment of abetment if person abetted does ...,Ditto,Ditto,Ditto,Ditto.
3,51.0,Liability of abettor when one act abetted and ...,Same as for offence intended to be abetted.,Ditto,Ditto,Ditto.
4,52.0,Abettor when liable to cumulative punishment f...,Same as for offence committed.,Ditto,Ditto,Ditto


In [40]:
df = df.drop(index=0,axis=0)
df = df.reset_index(drop=True)

In [41]:
import numpy as np
df.replace({'Ditto': np.nan, 'Ditto.': np.nan}, inplace=True)
df = df.ffill(inplace=True)

In [20]:
# drop junk rows — keep only rows 345 and 346
df_table2 = df.iloc[345:347].copy()

# reset index
df_table2 = df_table2.reset_index(drop=True)

# rename columns properly
df_table2.columns = ['Punishment_Category', 'Offence', 'Description', 
                     'Cognizable', 'Bailable', 'Court']

# merge into single chunk
chunk_text = " | ".join([
    f"{row['Punishment_Category']}: "
    f"Cognizable: {row['Cognizable']}. "
    f"Bailable: {row['Bailable']}. "
    f"Court: {row['Court']}"
    for _, row in df_table2.iterrows()
])

other_laws_chunk = {
    "chunk_text": chunk_text,
    "parent_id":  None,
    "metadata": {
        "act":     "BNSS",
        "section": "other_laws",
        "type":    "table"
    }
}

print(chunk_text)

If punishable with death, imprisonment for life, or imprisonment for more than 7 years If punishable with imprisonment for 3 years and upwards but not more than 7 years: Cognizable: Cognizable Ditto. Bailable: Non-bailable Ditto. Court: Court of Session. Magistrate of the first class. | If punishable with imprisonment for less than 3 years or with fine only.: Cognizable: Non- cognizable. Bailable: Bailable. Court: Any Magistrate.


In [23]:
# remove last junk rows (343 onwards)
df_clean = df.iloc[:343].copy()
df_clean = df_clean.reset_index(drop=True)

In [32]:
# clean section column first before applying
df_clean['Section'] = df_clean['Section'].astype(str)

def row_to_chunk(row):
    return (
        f"Section {row['Section']}: "
        f"{row['Offence']}. "
        f"Punishment: {row['Punishment']}. "
        f"Cognizable: {row['Cognizable or non- cognizable']}. "
        f"Bailable: {row['Bailable or Non- bailable']}. "
        f"Triable by: {row['By what Court triable']}"
    )

df_clean['chunk_text'] = df_clean.apply(row_to_chunk, axis=1)

### SECTIONS CHUNKING

In [2]:
# CORRECTING THE STRUCTURE OF THE MD FILE

import re

with open(r'D:\LegalSaathi AI\src\notebooks\docs\bnss\bnss_sections.md', 'r', encoding='utf-8') as f:
    text = f.read()

# strict: only matches "52." or "35." style — digits + dot, at start of line
text = re.sub(r'(?m)^(\d+)\.\s', r'### \1\n\n', text)


# converrt  ## CHAPTER -> # CHAPTER

text = re.sub(r'^#{1,6}\s*(CHAPTER\s+[IVXLCDM\d]+.*)$', r'# \1', text, flags=re.MULTILINE | re.IGNORECASE)

# create proper ## title for the sections
text = re.sub(
    r'(# CHAPTER [IVXLCDM]+\n\n+)([A-Z][A-Z\s,]+)',
    r'\1## \2',
    text
)

with open(r'D:\LegalSaathi AI\src\notebooks\docs\bnss\final_bnss_sections.md', 'w', encoding='utf-8') as f:
    f.write(text)

print("Done")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\LegalSaathi AI\\src\\notebooks\\docs\\bnss\\bnss_sections.md'

In [7]:
with open(r'D:\LegalSaathi AI\src\notebooks\docs\bnss\final_bnss_sections.md',encoding='utf8') as f:
    md_file = f.read()

from langchain_text_splitters import MarkdownHeaderTextSplitter

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ('#','chapter'),
        ('##','chapter_title'),
        ('###','section_data')
    ],strip_headers=True
)

docs =splitter.split_text(md_file)

In [ ]:

import tiktoken
import math

encoding = tiktoken.get_encoding("cl100k_base")

#count tokens 
def count_tokens(text):
    return len(encoding.encode(text))

def decide_chunks_size(text):
    num_tokens_split_criteria = 0
    
    tokens_length = count_tokens(text)
    
    # example 10 range 
    
    for i in range(1,10):
        if tokens_length / i < 400:
            #print(tokens_length / i)
            num_tokens_split_criteria = i
            break
        
    return num_tokens_split_criteria

def make_child_chunks(doc,num_tokens_split_criteria):
    
    childrens_chunks_list = []
    final_chunks = []

    sentence_chunks = doc.split('\n')
    
    if num_tokens_split_criteria > 1:

        broken_shapes = math.floor(len(sentence_chunks) / num_tokens_split_criteria)
        for i in range(0,len(sentence_chunks),broken_shapes):
            
            childrens_chunks_list.append(sentence_chunks[i: i+ broken_shapes])
        
        for chunk in childrens_chunks_list:
            
            children = ' '.join(chunk)
            
            children = children.replace('  ',' ')
            children = children.replace('-   ',' ')
            children = children.replace('-  ',' ')
            
            
            final_chunks.append(children)    
                
    else:
        
        final_chunks = [doc]

    # clean the doc before returining since its going in parent data as parent
    
    doc = doc.replace('\n','')
    doc = doc.replace('_','')
    
    
    return doc , final_chunks

In [1]:
docs

NameError: name 'docs' is not defined

In [ ]:
# PARENT-CHILD CHUNKING AND SAVE THE DATA INTO DB 

In [ ]:
parent_data = {}
